In [3]:
from pathlib import Path
import pickle
from pprint import pprint

# Find the file in the repo/workspace
candidates = list(Path(".").glob("data/THuman_2.0_smplx_paras/0001/smplx_param.pkl"))
if not candidates:
    raise FileNotFoundError("Cannot find THuamn_2.0_smplx_paras/smplx_param.pkl")

pkl_path = candidates[0]
print(f"Loading: {pkl_path}")

with open(pkl_path, "rb") as f:
    data = pickle.load(f)
    for key, value in data.items():
        print(f"{key}: {type(value), value.shape if hasattr(value, 'shape') else len(value)}")

Loading: data/THuman_2.0_smplx_paras/0001/smplx_param.pkl
betas: (<class 'numpy.ndarray'>, (1, 10))
global_orient: (<class 'numpy.ndarray'>, (1, 3))
body_pose: (<class 'numpy.ndarray'>, (1, 63))
left_hand_pose: (<class 'numpy.ndarray'>, (1, 12))
right_hand_pose: (<class 'numpy.ndarray'>, (1, 12))
jaw_pose: (<class 'numpy.ndarray'>, (1, 3))
leye_pose: (<class 'numpy.ndarray'>, (1, 3))
reye_pose: (<class 'numpy.ndarray'>, (1, 3))
expression: (<class 'numpy.ndarray'>, (1, 10))
scale: (<class 'numpy.ndarray'>, (1,))
translation: (<class 'trimesh.caching.TrackedArray'>, (3,))


In [12]:
print(data['global_orient'])

[[-0.23826408  1.3167884  -0.18078895]]


In [13]:
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.tri as mtri

def render_front_view(verts, faces, save_path=None, title=None, figsize=(4, 6)):
    """Orthographic front-view render (project to XY plane)."""
    tri = mtri.Triangulation(verts[:, 0], verts[:, 1], faces)
    face_depth = verts[faces][:, :, 2].mean(axis=1)

    fig, ax = plt.subplots(figsize=figsize)
    ax.tripcolor(tri, facecolors=face_depth, shading="flat", cmap="gray", edgecolors="none")
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title)

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200, bbox_inches="tight", pad_inches=0)
        plt.close(fig)
    else:
        plt.show()

# Base tensors (reuse existing model/data)
to_t = lambda x: torch.tensor(x, dtype=torch.float32)
base_y = float(data["global_orient"][0, 1])

render_dir = Path("renders_self_rotation")
angles_deg = np.arange(0, 360, 30)

for deg in angles_deg:
    # rotate around global Y (additive on top of current orientation)
    g_orient = data["global_orient"].copy()
    g_orient[0, 1] = base_y + np.deg2rad(deg)

    with torch.no_grad():
        out = model(
            betas=to_t(data["betas"]),
            global_orient=to_t(g_orient),
            body_pose=to_t(data["body_pose"]),
            left_hand_pose=to_t(data["left_hand_pose"]),
            right_hand_pose=to_t(data["right_hand_pose"]),
            jaw_pose=to_t(data["jaw_pose"]),
            leye_pose=to_t(data["leye_pose"]),
            reye_pose=to_t(data["reye_pose"]),
            expression=to_t(data["expression"]),
            return_verts=True,
        )

    verts_i = out.vertices[0].cpu().numpy()
    if "scale" in data:
        verts_i = verts_i * float(data["scale"][0])
    if "translation" in data:
        verts_i = verts_i + data["translation"].reshape(1, 3)

    render_front_view(
        verts_i,
        faces,
        save_path=render_dir / f"front_{deg:03d}.png",
        title=f"Y rotation: {deg}°",
    )

print(f"Saved {len(angles_deg)} renders to: {render_dir.resolve()}")

Saved 12 renders to: /Users/lemon/Documents/TUD/Thesis/avatar-benchmark/renders_self_rotation


In [16]:
# Build a neutral T-pose-style parameter set, then rotate around global Y.
# (Global orientation is zero, body/hand/face poses are neutral.)
canonical_orient = np.zeros_like(data["global_orient"], dtype=np.float32)
canonical_body_pose = np.zeros_like(data["body_pose"], dtype=np.float32)
canonical_left_hand_pose = np.zeros_like(data["left_hand_pose"], dtype=np.float32)
canonical_right_hand_pose = np.zeros_like(data["right_hand_pose"], dtype=np.float32)
canonical_jaw_pose = np.zeros_like(data["jaw_pose"], dtype=np.float32)
canonical_leye_pose = np.zeros_like(data["leye_pose"], dtype=np.float32)
canonical_reye_pose = np.zeros_like(data["reye_pose"], dtype=np.float32)
canonical_expression = np.zeros_like(data["expression"], dtype=np.float32)

render_dir = Path("renders_tpose_y_rotation")
render_dir.mkdir(parents=True, exist_ok=True)

for deg in angles_deg:
    g_orient = canonical_orient.copy()
    g_orient[0, 1] = np.deg2rad(deg)

    with torch.no_grad():
        out = model(
            betas=torch.tensor(data["betas"], dtype=torch.float32),
            global_orient=torch.tensor(g_orient, dtype=torch.float32),
            body_pose=torch.tensor(canonical_body_pose, dtype=torch.float32),
            left_hand_pose=torch.tensor(canonical_left_hand_pose, dtype=torch.float32),
            right_hand_pose=torch.tensor(canonical_right_hand_pose, dtype=torch.float32),
            jaw_pose=torch.tensor(canonical_jaw_pose, dtype=torch.float32),
            leye_pose=torch.tensor(canonical_leye_pose, dtype=torch.float32),
            reye_pose=torch.tensor(canonical_reye_pose, dtype=torch.float32),
            expression=torch.tensor(canonical_expression, dtype=torch.float32),
            return_verts=True,
        )

    verts_i = out.vertices[0].cpu().numpy()
    if "scale" in data:
        verts_i = verts_i * float(data["scale"][0])
    if "translation" in data:
        verts_i = verts_i + data["translation"].reshape(1, 3)

    render_front_view(
        verts_i,
        faces,
        save_path=render_dir / f"front_{deg:03d}.png",
        title=f"T-pose + Y rotation: {deg}°",
    )

print(f"Saved {len(angles_deg)} renders to: {render_dir.resolve()}")

Saved 12 renders to: /Users/lemon/Documents/TUD/Thesis/avatar-benchmark/renders_tpose_y_rotation


In [6]:
import torch
from pathlib import Path
import smplx

# 1) Find an SMPL-X model file (neutral first, then any SMPLX_*.npz)
model_candidates = list(Path(".").rglob("SMPLX_NEUTRAL.npz"))
if not model_candidates:
    model_candidates = list(Path(".").rglob("SMPLX_*.npz"))
if not model_candidates:
    raise FileNotFoundError(
        "Cannot find SMPL-X model file (e.g., SMPLX_NEUTRAL.npz). "
        "Please place it in the workspace."
    )

model_file = model_candidates[0]
print(f"Using SMPL-X model: {model_file}")

# 2) Build SMPL-X model
num_betas = data["betas"].shape[1]
num_expr = data["expression"].shape[1]
hand_dim = data["left_hand_pose"].shape[1]

# THuman params can store hands either as full axis-angle (45) or PCA coefficients (e.g., 12)
if hand_dim == 45:
    use_pca = False
    num_pca_comps = 12  # ignored when use_pca=False
else:
    use_pca = True
    num_pca_comps = hand_dim

print(f"Hand pose dim: {hand_dim}, use_pca={use_pca}, num_pca_comps={num_pca_comps}")

model = smplx.SMPLX(
    model_path=str(model_file),
    num_betas=num_betas,
    num_expression_coeffs=num_expr,
    use_pca=use_pca,
    num_pca_comps=num_pca_comps,
    flat_hand_mean=False
).eval()

# 3) Convert pkl arrays to torch tensors
to_t = lambda x: torch.tensor(x, dtype=torch.float32)

with torch.no_grad():
    output = model(
        betas=to_t(data["betas"]),
        global_orient=to_t(data["global_orient"]),
        body_pose=to_t(data["body_pose"]),
        left_hand_pose=to_t(data["left_hand_pose"]),
        right_hand_pose=to_t(data["right_hand_pose"]),
        jaw_pose=to_t(data["jaw_pose"]),
        leye_pose=to_t(data["leye_pose"]),
        reye_pose=to_t(data["reye_pose"]),
        expression=to_t(data["expression"]),
        return_verts=True
    )

verts = output.vertices[0].cpu().numpy()
faces = model.faces

# 4) Apply scale + translation from smplx_param (for demonstration)
if "scale" in data:
    verts = verts * float(data["scale"][0])
if "translation" in data:
    verts = verts + data["translation"].reshape(1, 3)

# 5) Save OBJ
out_obj = Path("demo_smplx_from_param.obj")
with open(out_obj, "w") as fp:
    for v in verts:
        fp.write(f"v {v[0]} {v[1]} {v[2]}\n")
    for f_idx in faces:
        # OBJ face index is 1-based
        fp.write(f"f {f_idx[0]+1} {f_idx[1]+1} {f_idx[2]+1}\n")

print(f"OBJ saved to: {out_obj.resolve()}")

Using SMPL-X model: models/smplx/SMPLX_NEUTRAL.npz
Hand pose dim: 12, use_pca=True, num_pca_comps=12
OBJ saved to: /Users/lemon/Documents/TUD/Thesis/avatar-benchmark/demo_smplx_from_param.obj


In [17]:
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree

def load_obj_vertices(obj_path):
    verts = []
    with open(obj_path, "r") as f:
        for line in f:
            if line.startswith("v "):
                _, x, y, z = line.strip().split()[:4]
                verts.append([float(x), float(y), float(z)])
    return np.asarray(verts, dtype=np.float32)

# Generated OBJ from the previous cell
generated_obj = load_obj_vertices(out_obj)
original_obj = load_obj_vertices(pkl_path.parent / "mesh_smplx.obj")

print(f"Generated shape: {generated_obj.shape}")
print(f"Original shape:  {original_obj.shape}")

if generated_obj.shape != original_obj.shape:
    print("Vertex counts differ, so they are not the same mesh topology.")
else:
    # 1) Strict index-wise check (same vertex order and positions)
    diff = np.abs(generated_obj - original_obj)
    l2 = np.linalg.norm(generated_obj - original_obj, axis=1)
    print("\n[Index-wise check]")
    print(f"Max abs diff: {diff.max():.8f}")
    print(f"Mean abs diff: {diff.mean():.8f}")
    print(f"Mean L2 diff per vertex: {l2.mean():.8f}")
    print("Same vertices (atol=1e-6):", np.allclose(generated_obj, original_obj, atol=1e-6))

    # 2) Order-agnostic point-set check (ignores vertex indexing)
    # If meshes look visually similar but index-wise check fails, this is the useful metric.
    tree_orig = cKDTree(original_obj)
    d_gen_to_orig, _ = tree_orig.query(generated_obj, k=1)

    tree_gen = cKDTree(generated_obj)
    d_orig_to_gen, _ = tree_gen.query(original_obj, k=1)

    print("\n[Order-agnostic set check]")
    print(f"gen -> orig NN mean: {d_gen_to_orig.mean():.8f}, max: {d_gen_to_orig.max():.8f}")
    print(f"orig -> gen NN mean: {d_orig_to_gen.mean():.8f}, max: {d_orig_to_gen.max():.8f}")
    print(f"gen -> orig NN < 1e-3: {(d_gen_to_orig < 1e-3).mean() * 100:.2f}%")
    print(f"orig -> gen NN < 1e-3: {(d_orig_to_gen < 1e-3).mean() * 100:.2f}%")

Generated shape: (10475, 3)
Original shape:  (10475, 3)

[Index-wise check]
Max abs diff: 0.56174952
Mean abs diff: 0.07740075
Mean L2 diff per vertex: 0.16819727
Same vertices (atol=1e-6): False

[Order-agnostic set check]
gen -> orig NN mean: 0.00260081, max: 0.02064594
orig -> gen NN mean: 0.00289715, max: 0.02133127
gen -> orig NN < 1e-3: 24.28%
orig -> gen NN < 1e-3: 24.35%
